## 4. cropped and image folder
- train_images
- test_images

## Circuits Dataset
- train
    - simple circuit
    - montreal circuit
    - montmelo circuit
- test
    - nurburgring circuit

In [1]:
import os
import glob
import torch
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import time

In [2]:
# CONSTANTs
CIRCUIT_LIST = [
    "simple_circuit",
    "montreal_circuit",
    "montmelo_circuit",
    "nurburgring_circuit"
]
TEST_CIRCUIT = "montmelo_circuit"
IMG_HEIGHT = 240

In [3]:
#input and output path
INPUT_BASE_DIR = ""

OUTPUT_BASE_DIR = "" 

In [4]:
# CREATE TRAIN AND TEST DIR INSIDE BASE OUTPUT DIR
os.makedirs(os.path.join(OUTPUT_BASE_DIR,'train'),exist_ok=True)
os.makedirs(os.path.join(OUTPUT_BASE_DIR,'test'),exist_ok=True)

### 🔁 Loop over all the circuits

In [ ]:
# start time
start_time = time.time()
error_images = 0


for circuit in CIRCUIT_LIST:
    # GET THE FULL IMAGE INPUT PATH IF CURRENT `circuit`
    image_path = os.path.join(INPUT_BASE_DIR,circuit,'images')
    
    # CREATE FOLDER/CHECK
    os.makedirs(image_path,exist_ok=True)

    # GET LIST OF ALL THE `circuit` IMAGES
    image_list = glob.glob(os.path.join(image_path,'*.png'))

    # Transform to tensor
    to_tensor = transforms.ToTensor()  # Converts PIL to [C,H,W] FloatTensor in [0,1]
    to_pil = transforms.ToPILImage()   # For visualization or debug

    # LOOP THROUGH THE IAMGES
    for path in image_list:
        try:
            # Load image with PIL and convert to RGB
            img = Image.open(path).convert('RGB')
    
            # To tensor and move to GPU
            tensor = to_tensor(img).unsqueeze(0).cuda()  # Shape: [1, 3, H, W]
    
            # Crop last 240 rows
            cropped_tensor = tensor[:, :, -IMG_HEIGHT:, :]  # Shape: [1, 3, 240, W]
    
            # Move to CPU and convert to numpy image (H, W, C) in uint8
            cropped_img_np = (cropped_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    
            # Save using OpenCV
            filename = os.path.basename(path)

            # divided path into train and test
            if circuit == TEST_CIRCUIT:
                save_path = os.path.join(OUTPUT_BASE_DIR,'test', filename)
            else: 
                save_path = os.path.join(OUTPUT_BASE_DIR,'train',filename)
                
            cv2.imwrite(save_path, cv2.cvtColor(cropped_img_np, cv2.COLOR_RGB2BGR))
    
            # print(f"Saved: {save_path}")
    
        except Exception as e:
            print(f"Error processing {path}: {e}")
            error_images += 1
            
    #  end of the loop

    
# end time
end_time = time.time()
elapsed = end_time - start_time

print(f"Failed to process {error_images} images.")
print(f"Elapsed time: {elapsed:.2f} seconds")